# Hämtar data till RAG-modellen

TestConnection för att se att API fungerar

In [ ]:
import json
import os
import requests
from dotenv import load_dotenv

# Test

In [5]:
load_dotenv()

TOKEN = os.getenv("THINKIFIC_API_ACCESS_TOKEN")
URL = os.getenv("THINKIFIC_GRAPHQL_URL")
SUBDOMAIN = os.getenv("THINKIFIC_SUBDOMAIN")

query = """
query TestConnection {
  __typename
}
"""

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "X-Thinkific-Subdomain": SUBDOMAIN
}

response = requests.post(
    URL,
    headers=headers,
    json={"query": query},
    timeout=30
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"data":{"__typename":"Query"},"extensions":{"rateLimit":{"cost":1,"remaining":1844,"resetAt":"2026-04-09T16:37:00.000Z","limit":2000}}}



In [22]:
load_dotenv()

TOKEN = os.getenv("THINKIFIC_API_ACCESS_TOKEN")
SUBDOMAIN = os.getenv("THINKIFIC_SUBDOMAIN")
URL = "https://api.thinkific.com/stable/graphql"

query = """
query GetLessonBasic {
  lesson(id: 73644304) {
    id
    title
    lessonType
    takeUrl
    content {
      __typename
      id
      contentType
    }
  }
}
"""

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "X-Thinkific-Subdomain": SUBDOMAIN
}

response = requests.post(
    URL,
    headers=headers,
    json={"query": query},
    timeout=30
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

Status: 200
{
  "data": {
    "lesson": {
      "id": "73644304",
      "title": " De grundläggande nautiska termerna för båtens delar",
      "lessonType": "TEXT",
      "takeUrl": "https://moresailing.thinkific.com/courses/take/ai-placeholder-3/texts/73644304-de-grundlaggande-nautiska-termerna-for-batens-delar",
      "content": {
        "__typename": "TextContent",
        "id": "19847536",
        "contentType": "TEXT"
      }
    }
  },
  "extensions": {
    "rateLimit": {
      "cost": 3,
      "remaining": 1952,
      "resetAt": "2026-04-09T17:40:00.000Z",
      "limit": 2000
    }
  }
}


In [25]:
load_dotenv()

TOKEN = os.getenv("THINKIFIC_API_ACCESS_TOKEN")
SUBDOMAIN = os.getenv("THINKIFIC_SUBDOMAIN")
URL = "https://api.thinkific.com/stable/graphql"

query = """
query GetLessonText {
  lesson(id: 73644304) {
    id
    title
    lessonType
    takeUrl
    chapter {
      id
      title
    }
    course {
      id
      name
      slug
    }
    content {
      __typename
      ... on TextContent {
        id
        contentType
        htmlDescription
      }
    }
  }
}
"""

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "X-Thinkific-Subdomain": SUBDOMAIN
}

response = requests.post(
    URL,
    headers=headers,
    json={"query": query},
    timeout=30
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

Status: 200
{
  "data": {
    "lesson": {
      "id": "73644304",
      "title": " De grundläggande nautiska termerna för båtens delar",
      "lessonType": "TEXT",
      "takeUrl": "https://moresailing.thinkific.com/courses/take/ai-placeholder-3/texts/73644304-de-grundlaggande-nautiska-termerna-for-batens-delar",
      "chapter": {
        "id": "14568721",
        "title": "Båtdelar och Terminologi"
      },
      "course": {
        "id": "3064200",
        "name": "Segling för Nybörjare",
        "slug": "ai-placeholder-3"
      },
      "content": {
        "__typename": "TextContent",
        "id": "19847536",
        "contentType": "TEXT",
        "htmlDescription": "<h3 style=\"box-sizing: border-box; margin-top: 1.25em; line-height: 1.75rem; font-size: 1.25rem; font-weight: 600; margin-bottom: 1em; color: inherit; font-family: DarwinPro, Helvetica, Arial, sans-serif;\"><span style=\"font-size: 16px;\">Nautiska termer f&ouml;r b&aring;tens delar</span></h3><ul style=\"box-sizin

## Raw import Segling för nybörjare

In [26]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("THINKIFIC_API_ACCESS_TOKEN")
SUBDOMAIN = os.getenv("THINKIFIC_SUBDOMAIN")
URL = "https://api.thinkific.com/stable/graphql"

COURSE_ID = 3064200

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "X-Thinkific-Subdomain": SUBDOMAIN
}

course_query_template = """
query GetCourseStructure {{
  course(id: {course_id}) {{
    id
    name
    slug
    curriculum {{
      chaptersCount
      lessonsCount
      chapters(first: 50) {{
        nodes {{
          id
          title
          position
          lessons(first: 100) {{
            nodes {{
              id
              title
              lessonType
            }}
          }}
        }}
      }}
    }}
  }}
}}
"""

lesson_query_template = """
query GetLessonText {{
  lesson(id: {lesson_id}) {{
    id
    title
    lessonType
    takeUrl
    chapter {{
      id
      title
    }}
    course {{
      id
      name
      slug
    }}
    content {{
      __typename
      ... on TextContent {{
        id
        contentType
        htmlDescription
      }}
    }}
  }}
}}
"""

def run_query(query: str) -> dict:
    response = requests.post(
        URL,
        headers=headers,
        json={"query": query},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise ValueError(json.dumps(data["errors"], indent=2, ensure_ascii=False))

    return data

def main():
    raw_dir = Path(f"data/raw/{COURSE_ID}")
    raw_dir.mkdir(parents=True, exist_ok=True)

    # 1. Hämta kursstruktur
    course_query = course_query_template.format(course_id=COURSE_ID)
    course_data = run_query(course_query)
    course = course_data["data"]["course"]

    with open(raw_dir / "course_structure.json", "w", encoding="utf-8") as f:
        json.dump(course_data, f, indent=2, ensure_ascii=False)

    print(f"Kurs: {course['name']}")

    # 2. Hämta alla lektioner
    for chapter in course["curriculum"]["chapters"]["nodes"]:
        for lesson in chapter["lessons"]["nodes"]:
            lesson_id = lesson["id"]
            lesson_query = lesson_query_template.format(lesson_id=lesson_id)
            lesson_data = run_query(lesson_query)
            lesson_obj = lesson_data["data"]["lesson"]
            content = lesson_obj.get("content") or {}

            output = {
                "course_id": lesson_obj["course"]["id"] if lesson_obj.get("course") else None,
                "course_name": lesson_obj["course"]["name"] if lesson_obj.get("course") else None,
                "course_slug": lesson_obj["course"]["slug"] if lesson_obj.get("course") else None,
                "chapter_id": lesson_obj["chapter"]["id"] if lesson_obj.get("chapter") else None,
                "chapter_title": lesson_obj["chapter"]["title"] if lesson_obj.get("chapter") else None,
                "lesson_id": lesson_obj["id"],
                "lesson_title": lesson_obj["title"].strip(),
                "lesson_type": lesson_obj["lessonType"],
                "take_url": lesson_obj["takeUrl"],
                "content_typename": content.get("__typename"),
                "content_id": content.get("id"),
                "content_type": content.get("contentType"),
                "html_description": content.get("htmlDescription")
            }

            with open(raw_dir / f"{lesson_id}.json", "w", encoding="utf-8") as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            print(f"Sparade lesson {lesson_id}: {lesson_obj['title'].strip()}")

    print("Export klar.")

if __name__ == "__main__":
    main()

Kurs: Segling för Nybörjare
Sparade lesson 73644304: De grundläggande nautiska termerna för båtens delar
Sparade lesson 63949181: Seglingsspråk
Sparade lesson 73644264: Olika segel
Sparade lesson 63949184: Så Sätter du Segel
Sparade lesson 63949186: Styrning av Båten
Export klar.


## Cleaning för Segling för Nybörjare

In [27]:
import json
from pathlib import Path
from bs4 import BeautifulSoup

COURSE_ID = 3064200

raw_dir = Path(f"data/raw/{COURSE_ID}")
cleaned_dir = Path(f"data/cleaned/{COURSE_ID}")
cleaned_dir.mkdir(parents=True, exist_ok=True)

for file_path in raw_dir.glob("*.json"):
    if file_path.name == "course_structure.json":
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    html = data.get("html_description") or ""
    text = BeautifulSoup(html, "html.parser").get_text("\n", strip=True)

    cleaned = {
        "course_id": data["course_id"],
        "course_name": data["course_name"],
        "course_slug": data["course_slug"],
        "chapter_id": data["chapter_id"],
        "chapter_title": data["chapter_title"],
        "lesson_id": data["lesson_id"],
        "lesson_title": data["lesson_title"],
        "lesson_type": data["lesson_type"],
        "take_url": data["take_url"],
        "text": text
    }

    with open(cleaned_dir / file_path.name, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

    print(f"Rensade {file_path.name}")

print("HTML -> text klart.")

Rensade 63949181.json
Rensade 63949184.json
Rensade 63949186.json
Rensade 73644264.json
Rensade 73644304.json
HTML -> text klart.
